In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(directory):
    """Process all PDF files in a directory."""
    all_documents = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(".pdf"):
                pdf_path = os.path.join(root, file)
                print(f"Loading: {file}...")
                loader = PyPDFLoader(pdf_path)
                docs = loader.load()
                all_documents.extend(docs)
                
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

C:\Users\M.ADARSH\AppData\Local\Temp\ipykernel_2704\4251085996.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading: agentic-ai.pdf...


C:\Users\M.ADARSH\AppData\Roaming\Python\Python313\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Loading: ai-ijirt.pdf...
Loading: rag.pdf...

Total documents loaded: 72


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:e76afa9)', 'creationdate': '', 'author': 'Mohamad Abou Ali; Fadi Dornaika', 'doi': 'https://doi.org/10.48550/arXiv.2510.25445', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Agentic AI: A Comprehensive Survey of Architectures, Applications, and Future Directions', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2510.25445v1', 'source': '../data\\pdf\\agentic-ai.pdf', 'total_pages': 46, 'page': 0, 'page_label': '1'}, page_content='Agentic AI: A Comprehensive Survey of\nArchitectures, Applications, and Future\nDirections\nMohamad Abou Ali1, 3, 4and Fadi Dornaika*1, 2\n1University of the Basque Country,2IKERBASQUE,3Lebanese\nInternational University (LIU),4The International University of Beirut,\nmohamad.abouali01@liu.edu.lb, fadi.dornaika@ehu.eus\nAbstract\nAgenti

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [11]:
chunks = split_documents(all_pdf_documents)

Split 72 documents into 335 chunks

Example chunk:
Content: Agentic AI: A Comprehensive Survey of
Architectures, Applications, and Future
Directions
Mohamad Abou Ali1, 3, 4and Fadi Dornaika*1, 2
1University of the Basque Country,2IKERBASQUE,3Lebanese
Internati...
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:e76afa9)', 'creationdate': '', 'author': 'Mohamad Abou Ali; Fadi Dornaika', 'doi': 'https://doi.org/10.48550/arXiv.2510.25445', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Agentic AI: A Comprehensive Survey of Architectures, Applications, and Future Directions', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2510.25445v1', 'source': '../data\\pdf\\agentic-ai.pdf', 'total_pages': 46, 'page': 0, 'page_label': '1'}


In [12]:
## embedding and vector store db

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self,texts: List[str])-> np.ndarray:

        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed
        Returns:
               numpy array of embeddings with shape (len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for{len(texts)} texts....")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings


## initilize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\M.ADARSH\AppData\Local\Temp\ipykernel_2704\2627331250.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


In [15]:
import os
import chromadb
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [16]:
chunks

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:e76afa9)', 'creationdate': '', 'author': 'Mohamad Abou Ali; Fadi Dornaika', 'doi': 'https://doi.org/10.48550/arXiv.2510.25445', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Agentic AI: A Comprehensive Survey of Architectures, Applications, and Future Directions', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2510.25445v1', 'source': '../data\\pdf\\agentic-ai.pdf', 'total_pages': 46, 'page': 0, 'page_label': '1'}, page_content='Agentic AI: A Comprehensive Survey of\nArchitectures, Applications, and Future\nDirections\nMohamad Abou Ali1, 3, 4and Fadi Dornaika*1, 2\n1University of the Basque Country,2IKERBASQUE,3Lebanese\nInternational University (LIU),4The International University of Beirut,\nmohamad.abouali01@liu.edu.lb, fadi.dornaika@ehu.eus\nAbstract\nAgenti

In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for335 texts....


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Generated embeddings with shape:(335, 384)
Adding 335 documents to vector store...
Successfully added 335 documents to vector store
Total documents in collection: 335


In [18]:
### rag retreiver pipline

In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [20]:
rag_retriever

In [22]:
rag_retriever.retrieve("What is artificial intelligence")

Retrieving documents for query: 'What is artificial intelligence'
Top K: 5, Score threshold: 0.0
Generating embeddings for1 texts....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape:(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_812049a7_169',
  'content': '© 202 3 IJRTI | Volume 8, Issue 4 | ISSN: 2456 -3315  \n  \nIJRTI230406 1 International Journal for Research Trends and Innovation ( www.ijrti.org ) 356 \n RESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS  \n \n \nProf. Neha Saini  \n \nAssistant Professor in Department of Computer Science & IT  \nSDAM College Dinanagar  \n \nABSTRACT - \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to \nthe similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that \nare biologically observable. While no consensual definition of Artificial Intelligence (AI) exists, AI is broadly characteriz ed \nas the study of computa tions that allow for perception, reason and action.  Today, the amount of data that is generated, by \nboth humans and machines, far outpaces humans’ ability to absorb, interpret, and make co

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)
llm


In [ ]:
def rag_simple(query, retriever, llm):
    # Retrieve relevant documents
    results = retriever.retrieve(query)
    context = "\n\n".join([doc["content"] for doc in results])
    
    # Generate answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content


In [ ]:
answer = rag_simple("What is artificial intelligence", rag_retriever, llm)
print(answer)
